# Análisis de Movimiento y Video
Este notebook muestra técnicas básicas para analizar movimiento en vídeo: detección por diferencia de frames y flujo óptico denso (Farneback).

In [1]:
# Librerías necesarias
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Video, display, Markdown

## Carga de vídeo (sube un archivo llamado `video.mp4` en el entorno de ejecución)

In [2]:
# Selección del vídeo
VIDEO_PATH = Path("captura.mp4")

try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        VIDEO_PATH = Path(next(iter(uploaded)))
except Exception:
    pass

print(f"Vídeo seleccionado: {VIDEO_PATH}")
print(f"¿Existe el archivo?: {VIDEO_PATH.exists()}")

Vídeo seleccionado: captura.mp4
¿Existe el archivo?: True


In [3]:
# Mostrar el vídeo (solo Colab o Jupyter con soporte HTML)
if VIDEO_PATH.exists():
    display(Video(str(VIDEO_PATH), embed=True))
else:
    print("No se ha encontrado el vídeo. Súbelo en la celda anterior o copia un archivo con ese nombre.")

## Actividad guiada (análisis local con OpenCV)
Estas celdas deben ejecutarse en un entorno local con acceso a ventana de OpenCV.

## Captura de vídeo desde cámara
Un flujo de trabajo razonable es grabar primero un vídeo corto con la cámara, guardarlo en disco y después aplicar sobre ese archivo las técnicas de análisis. Así puedes repetir pruebas con el mismo material sin depender de la captura en tiempo real.

In [4]:
# Captura un vídeo desde la cámara y guárdalo en disco (ejecutar localmente)
camera_index = 0
output_path = "captura.mp4"
fps = 20.0
frame_size = (640, 480)

cap = cv2.VideoCapture(camera_index)
if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara")

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, frame_size)

print("Grabando vídeo. Pulsa 'q' para terminar.")

while True:
    ret, frame = cap.read()
    if not ret or frame is None:
        break

    frame = cv2.resize(frame, frame_size)
    out.write(frame)
    cv2.imshow("Grabacion desde camara", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
out.release()
cv2.destroyAllWindows()
print(f"Vídeo guardado en: {output_path}")

Grabando vídeo. Pulsa 'q' para terminar.
Vídeo guardado en: captura.mp4


In [5]:
# Detección de movimiento por diferencia de frames (ejecutar localmente)
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

ret1, frame1 = cap.read()
ret2, frame2 = cap.read()

if not ret1 or frame1 is None:
    cap.release()
    raise ValueError("No se pudo leer el primer frame del vídeo.")

if not ret2 or frame2 is None:
    cap.release()
    raise ValueError("No se pudo leer el segundo frame del vídeo.")

while True:
    diff = cv2.absdiff(frame1, frame2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
    dilated = cv2.dilate(thresh, None, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        if cv2.contourArea(contour) < 1000:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame1, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # VideoCapture solo lee frames; la visualización la hace imshow en una ventana local
    cv2.imshow('Movimiento', frame1)

    frame1 = frame2
    ret2, frame2 = cap.read()
    if not ret2 or frame2 is None:
        break

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

In [9]:
# Flujo óptico denso (Farneback, ejecutar localmente)
cap = cv2.VideoCapture(str(VIDEO_PATH))

if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el vídeo: {VIDEO_PATH}")

ret, frame1 = cap.read()
if not ret or frame1 is None:
    cap.release()
    raise ValueError("No se pudo leer el primer frame del vídeo.")

prvs = cv2.cvtColor(frame1, cv2.COLOR_BGR2GRAY)
hsv = np.zeros_like(frame1)
hsv[..., 1] = 255

while True:
    ret, frame2 = cap.read()
    if not ret or frame2 is None:
        break

    next = cv2.cvtColor(frame2, cv2.COLOR_BGR2GRAY)
    flow = cv2.calcOpticalFlowFarneback(prvs, next, None, 0.5, 3, 15, 3, 5, 1.2, 0)
    mag, ang = cv2.cartToPolar(flow[..., 0], flow[..., 1])
    hsv[..., 0] = ang * 180 / np.pi / 2
    hsv[..., 2] = cv2.normalize(mag, None, 0, 255, cv2.NORM_MINMAX)
    bgr = cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)
    cv2.imshow('Flujo óptico', bgr)

    if cv2.waitKey(30) & 0xFF == ord('q'):
        break

    prvs = next

cap.release()
cv2.destroyAllWindows()

KeyboardInterrupt: 

## Análisis en vivo con la cámara
Si el ordenador tiene suficiente rendimiento, puedes aplicar la detección directamente sobre la señal de la cámara. Conceptualmente es lo mismo que antes, pero sin pasar por un archivo intermedio.

In [6]:
# Detección de movimiento en vivo desde la cámara (ejecutar localmente)
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("No se pudo abrir la cámara")

ret1, frame1 = cap.read()
ret2, frame2 = cap.read()

if not ret1 or frame1 is None or not ret2 or frame2 is None:
    cap.release()
    raise ValueError("No se pudieron leer los primeros frames de la cámara.")

while True:
    diff = cv2.absdiff(frame1, frame2)
    gray = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
    _, thresh = cv2.threshold(gray, 30, 255, cv2.THRESH_BINARY)
    dilated = cv2.dilate(thresh, None, iterations=2)
    contours, _ = cv2.findContours(dilated, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    for contour in contours:
        if cv2.contourArea(contour) < 1000:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        cv2.rectangle(frame1, (x, y), (x + w, y + h), (0, 255, 0), 2)

    cv2.imshow("Movimiento en vivo", frame1)

    frame1 = frame2
    ret2, frame2 = cap.read()
    if not ret2 or frame2 is None:
        break

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
# Actividad
Markdown('''
**Actividad sugerida**

1. Graba primero un vídeo corto con la cámara o usa uno ya existente.
2. Aplica sobre ese archivo la detección por diferencia de frames y el flujo óptico.
3. Ajusta los umbrales y el área mínima de detección de movimiento.
4. Si tu equipo responde bien, prueba la detección directamente en vivo con la cámara.
5. (Avanzado) Compara resultados con un algoritmo de seguimiento como CSRT.
''')